<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Feature Engineering

---

### About
Explain what feature engineering is, why it is important, and how to do it.

### Learning Objectives
- Explain the reasons for applying feature engineering to the data when modeling.
- Identify the relevant methods to apply to categorical data.
- Understand how and when to group continuous data into bins.
- Explain the reasons to transform your continuous data when modeling.
- Create transformers with scikit-learn to develop a pipeline of transformations for your data.

### Notebook Guide
1. What is Feature Engineering?
2. Working with Categorical Data
     - Label Encoding
     - One-Hot Encoding
3. Binning Continuous Variables
4. Transforming Continuous Variables
5. Building Feature Engineering Pipelines
6. Conclusions and Takeaways

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# These are needed for the second part of the lesson (Transforming Continuous Variables and Pipelines)
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.preprocessing import PowerTransformer, QuantileTransformer, KBinsDiscretizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

import warnings
warnings.filterwarnings('ignore')

# What is Feature Engineering?

Feature engineering is the process of transforming raw data into features that better represent the underlying problem to the predictive models, resulting in improved model accuracy on unseen data.

Think of it as preparing your ingredients before cooking - you wouldn't throw whole vegetables into a soup without chopping them first!

## The Data

We'll use a housing data set that contains various features about houses and their sale prices. This is perfect for demonstrating different feature engineering techniques.

In [ ]:
# Load the housing data
# For this demo, we'll create a synthetic data set that mimics real housing data
np.random.seed(42)

n_samples = 1000

# Create synthetic housing data
housing_data = {
    'sqft': np.random.normal(2000, 500, n_samples),
    'bedrooms': np.random.choice([1, 2, 3, 4, 5], n_samples, p=[0.1, 0.2, 0.4, 0.25, 0.05]),
    'bathrooms': np.random.choice([1, 1.5, 2, 2.5, 3, 3.5, 4], n_samples, p=[0.15, 0.1, 0.3, 0.15, 0.2, 0.05, 0.05]),
    'neighborhood': np.random.choice(['Downtown', 'Suburb', 'Rural', 'Waterfront'], n_samples, p=[0.3, 0.5, 0.15, 0.05]),
    'year_built': np.random.randint(1950, 2023, n_samples),
    'garage': np.random.choice(['None', 'Attached', 'Detached'], n_samples, p=[0.2, 0.6, 0.2]),
    'condition': np.random.choice(['Poor', 'Fair', 'Good', 'Excellent'], n_samples, p=[0.1, 0.2, 0.5, 0.2]),
    'income_area': np.random.exponential(50000, n_samples) + 30000
}

housing = pd.DataFrame(housing_data)

# Make sqft positive
housing['sqft'] = np.abs(housing['sqft'])

# Create a realistic price based on features
price_base = (housing['sqft'] * 150 + 
              housing['bedrooms'] * 10000 + 
              housing['bathrooms'] * 8000 + 
              (2023 - housing['year_built']) * -500 +
              housing['income_area'] * 2)

# Add neighborhood effects
neighborhood_effects = {'Downtown': 50000, 'Waterfront': 80000, 'Suburb': 20000, 'Rural': -10000}
for neighborhood, effect in neighborhood_effects.items():
    price_base += (housing['neighborhood'] == neighborhood) * effect

# Add some noise
housing['price'] = price_base + np.random.normal(0, 20000, n_samples)
housing['price'] = np.abs(housing['price'])  # Ensure positive prices

Let's take a quick look at our data types and see what we're working with:

---

# Working with Categorical Data

Categorical variables are variables that take on a limited number of distinct values. Most machine learning algorithms can't handle categorical data directly - they need numbers!

## Identifying Categorical Variables

Let's identify the unique values in each column for all features in our data set that are categorical:

In [ ]:
# Look at unique values in categorical columns


## Label Encoding

Label encoding converts categorical labels into numbers. This works well for ordinal data (where order matters).

In [ ]:
# Create a mapping for condition to make it ordinal


In [ ]:
# Check the relationship between encoded condition and price


<details><summary>Why is label encoding appropriate for 'condition' but not for 'neighborhood'?</summary>
Condition has a natural order (Poor < Fair < Good < Excellent), while neighborhood categories don't have a meaningful order.
</details>

## One-Hot Encoding

One-hot encoding creates binary columns for each category. This is better for nominal data (no natural order).

In [ ]:
# One-hot encode the neighborhood variable


# Fit and transform


# Get feature names


# Create DataFrame


In [ ]:
# Combine with original data


## Using `pandas` for Quick Encoding

`pandas` provides convenient methods for encoding:

In [ ]:
# Use pandas get_dummies for one-hot encoding


---

#### To the slides!

---


# Binning Continuous Variables

Sometimes it's useful to convert continuous variables into categorical bins. This can help capture non-linear relationships and make models more interpretable.

## When to Bin Continuous Variables

- When you suspect non-linear relationships
- To reduce the impact of outliers
- To make model interpretations easier
- When domain knowledge suggests natural breakpoints

## Manual Binning with `pandas.cut()`

Let's bin the house `age` into categories:

In [ ]:
# Create an age variable first


In [ ]:
# Create age bins

# Plot the distribution


In [ ]:
# Check price by age category


## Equal-Frequency (quantile-based) Binning with `pandas.qcut()`

Sometimes we want equal-sized bins rather than equal-width bins:

In [ ]:
# Create income quartiles


# Check the actual income ranges for each quartile


In [ ]:
# Plot income vs price by quartile


## An alternative way to do it: use `sklearn`'s `KBinsDiscretizer`

`scikit-learn` provides a more flexible binning tool:

In [ ]:
# Use KBinsDiscretizer on square footage


# Fit and transform


# Add to housing dataframe


# Check the bin edges


In [ ]:
# Compare binned vs continuous sqft relationship with price


# Continuous relationship


# Binned relationship



---

#### To the slides!

---

# Transforming Continuous Variables

Many machine learning algorithms work better when features are on similar scales or follow certain distributions.

## Why Transform?

1. **Scale differences**: Some features might be in thousands, others in decimals
2. **Distribution shape**: Many algorithms assume normal distributions
3. **Algorithm requirements**: Some algorithms are sensitive to scale (e.g., KNN, SVM)
4. **Convergence**: Neural networks often train faster with scaled inputs

## Visualizing Our Continuous Variables

Let's look at the distributions first:

In [ ]:
# Plot distributions of continuous variables


## Scaling Transformations

### StandardScaler (Z-score normalization)
Transforms data to have mean=0 and std=1

In [ ]:
# Apply StandardScaler to sqft


# Add to dataframe


In [ ]:
# Plot before and after scaling


### MinMaxScaler
Scales data to a fixed range (usually 0-1)

In [ ]:
# Apply MinMaxScaler to income_area


# Add to dataframe


In [ ]:
# Plot before and after MinMax scaling


## Distribution Transformations

### Log Transformation
Good for right-skewed data

In [ ]:
# Apply log transformation to income_area (which is right-skewed)


In [ ]:
# Plot before and after log transformation


### `PowerTransformer` (Box-Cox and Yeo-Johnson)
Automatically finds the best power transformation

In [ ]:
# Use PowerTransformer on income_area


# Add to dataframe


In [ ]:
# Plot before and after power transformation


### `QuantileTransformer`
Maps to a uniform or normal distribution

In [ ]:
# Use QuantileTransformer

# Add to dataframe


In [ ]:
# Compare all transformations


---

#### To the slides!

---

# Building Feature Engineering Pipelines

In real projects, you'll want to apply multiple transformations systematically. Pipelines help us do this cleanly and avoid data leakage.

## Why Use Pipelines?

1. **Prevents data leakage**: Transformations are fit only on training data
2. **Reproducibility**: Same transformations applied consistently
3. **Cleaner code**: All preprocessing in one place
4. **Easy deployment**: Transform new data the same way

## Setting up Our Data

First, let's prepare our features and target:

In [ ]:
# Define our feature columns and target


In [ ]:
# Split into train and test sets


## Creating Column-Specific Transformers

Different columns need different transformations. We can handle this with `ColumnTransformer`:

In [ ]:
# Define which columns are numeric and which are categorical


In [ ]:
# Create transformers for each type
# Numeric transformer: scale the features


# Categorical transformer: one-hot encode


In [ ]:
# Combine transformers with ColumnTransformer


## Creating a Complete Pipeline

Now let's combine our preprocessing with a model:

In [ ]:
# Create a complete pipeline


In [ ]:
# Fit the pipeline


## Evaluating Our Pipeline

Let's see how our feature engineering helps our model:

In [ ]:
# Make predictions


In [ ]:
# Evaluate performance


## Comparing with a Baseline

Let's compare this to a model without feature engineering:

In [ ]:
# Create a simple baseline model with just numeric features


# Simple linear regression without preprocessing


# Make predictions


# Evaluate baseline


In [ ]:
# Compare models


<details><summary>Which model performs better and why?</summary>
The engineered model should perform better because:
1. We included categorical information that was ignored in the baseline
2. We properly scaled the features
3. We handled different data types appropriately
</details>

## Advanced Pipeline Features

### Custom Transformers

Sometimes you need custom transformations:

In [ ]:
# Create a custom transformer that adds house age


In [ ]:
# Use the custom transformer in a pipeline
# First, add year_built to our features


# Split again


# Update numeric features to include age


# Update preprocessor


# Create pipeline with custom transformer


### Handling Missing Data

Let's add some missing data and see how to handle it:

In [ ]:
# Create some missing data for demonstration


# Randomly set 5% of sqft values to NaN


In [ ]:
# Add imputation to our pipeline
# Numeric transformer with imputation

# Categorical transformer with imputation

# Updated preprocessor with imputation

# Pipeline with imputation

# Fit on data with missing values


# Conclusions and Takeaways

* **Feature engineering is crucial** for model performance - sometimes more important than the choice of algorithm!
* **Different data types need different treatments**: categorical vs. continuous variables require different encoding strategies.
* **Scaling matters** for many algorithms, especially distance-based ones.
* **Pipelines are essential** for reproducible, leak-free preprocessing.
* **Domain knowledge** often guides the best feature engineering choices.
* **Always validate** your transformations - plot before and after!

### Next Steps

Practice these techniques on your own data sets! Remember:
- Start simple and add complexity gradually
- Always split your data before engineering features
- Document your transformations for reproducibility
- Experiment with different approaches and measure their impact